# Notebook 1: Data Preprocessing and EDA

**Objective:** Transform the 600,000 records raw dataset from an "exercise-per-row" structure to a "routine-per-row" structure. This transformation handles the data aggregation challenge to ensure the correct input tensor dimensionality for the subsequent Keras DNN model.

*Execution context: This notebook should be executed locally.*

In [2]:
%pip install pandas numpy

In [3]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../') # allow imports from src folder

# Set pandas display options for easier analysis
pd.set_option('display.max_columns', None)

## 1. Load Raw Dataset

Load the `.csv` containing the exercise-level records.

In [35]:
import os
import sys
import pandas as pd
import numpy as np

# Determine the base path depending on our current working directory
base_path = '../' if os.path.basename(os.getcwd()) == 'notebooks' else './'
sys.path.append(base_path) # allow imports from src folder

pd.set_option('display.max_columns', None)

In [36]:
raw_data_path = os.path.join(base_path, 'data/raw/dataset.csv')
df_raw = pd.read_csv(raw_data_path)

# Display basic statistics and shape
print(f"Initial shape: {df_raw.shape}")
display(df_raw.head())

Initial shape: (605033, 16)


,title,description,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,created,last_edit
0,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Knee-to-wall ankle dorsiflexion test,3.0,-180.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
1,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Banded ankle distractions,3.0,-18.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
2,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Slant board calf stretch,3.0,-18.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
3,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,Seated Tibialis Raise,3.0,15.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00
4,Weightlifting Mobility program,This program is designed to help athletes succ...,"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Garage Gym,7.0,50.0,1.0,1.0,16.0,90/90 Hip Rotations,3.0,8.0,6.0,2025-03-09 04:11:00,2025-06-23 03:31:00


In [37]:
# Check unique values per column
print("Unique elements per column:")
display(df_raw.nunique())

# check for missing values
# 1. Count standard null values (NaN, None)
null_counts = df_raw.isnull().sum()

# 2. Count empty list representations (read as strings from CSV)
empty_list_counts = (df_raw == '[]').sum()

# 3. Combine them to see the total missing/empty values per column
total_missing = null_counts + empty_list_counts

print("Total missing or '[]' values per column:")
display(total_missing)

Unique elements per column:


title                  2598
description            2530
level                    47
goal                    250
equipment                 4
program_length           18
time_per_workout         19
week                     18
day                      10
number_of_exercises      24
exercise_name          3213
sets                     18
reps                    205
intensity                11
created                2684
last_edit               623
dtype: int64

Total missing or '[]' values per column:


title                     0
description             819
level                  2936
goal                   2936
equipment                16
program_length           16
time_per_workout          0
week                      0
day                       0
number_of_exercises       0
exercise_name             0
sets                      0
reps                      0
intensity                 0
created                  16
last_edit                41
dtype: int64

In [38]:
# Get an array of all unique exercise names
unique_exercises = df_raw['exercise_name'].dropna().unique()

# Print the total count
print(f"Total unique exercises: {len(unique_exercises)}\n")

# Display the full list
display(unique_exercises)

Total unique exercises: 3213



<StringArray>
[   'Knee-to-wall ankle dorsiflexion test',
               'Banded ankle distractions',
                'Slant board calf stretch',
                   'Seated Tibialis Raise',
                     '90/90 Hip Rotations',
   'Pigeon Stretch with Thoracic Rotation',
                           'Hip Airplanes',
             'Quadruped T-Spine Rotations',
                     'Hanging Lat Stretch',
              'Banded Shoulder Dislocates',
 ...
                'Dual Handle Lat Pulldown',
             'Decline Machine Chest Press',
        'Super ROM Lateral Dumbbell Raise',
             'Medicine Ball Russian Twist',
             'Smith Machine Reverse Lunge',
            'Super ROM Overhead Cable Row',
                     'Dumbbell Calf Jumps',
         'Rear Delt 45 Degrees Cable Flye',
 'Triceps Diverging Pressdown (Long Rope)',
                  'Core Work (You Choose)']
Length: 3213, dtype: str

In [5]:
# 1. Filter the dataset for Garage Gym
garage_gym_df = df_raw[df_raw['equipment'] == 'Garage Gym']

# 2. Extract unique exercise names from the filtered data
garage_exercises = garage_gym_df['exercise_name'].dropna().unique()

# Print the count and display the list
print(f"Total Garage Gym exercises: {len(garage_exercises)}\n")
display(garage_exercises)

Total Garage Gym exercises: 1128



<StringArray>
[ 'Knee-to-wall ankle dorsiflexion test',
             'Banded ankle distractions',
              'Slant board calf stretch',
                 'Seated Tibialis Raise',
                   '90/90 Hip Rotations',
 'Pigeon Stretch with Thoracic Rotation',
                         'Hip Airplanes',
           'Quadruped T-Spine Rotations',
                   'Hanging Lat Stretch',
            'Banded Shoulder Dislocates',
 ...
         'Bent Knee Standing Calf Raise',
                     '21 Curl (Barbell)',
                           'Waiter Curl',
          'Standing Leg Raise with Band',
                    'Leg Curl with Band',
               'Leg Extension with Band',
                             'Yates Row',
                               '1km Row',
                        'Band Push-down',
                            'Ghd Sit Up']
Length: 1128, dtype: str

In [39]:
# Create a new dataframe without 'Garage Gym' rows, leaving df_raw untouched
df_filtered = df_raw[df_raw['equipment'] != 'Garage Gym'].copy()

# Print both shapes to verify your original dataset is safe!
print(f"Original shape (df_raw): {df_raw.shape}")
print(f"Filtered shape (df_filtered): {df_filtered.shape}")

Original shape (df_raw): (605033, 16)
Filtered shape (df_filtered): (507947, 16)


In [40]:
# Check unique values per column
print("Unique elements per column:")
display(df_filtered.nunique())

# check for missing values
# 1. Count standard null values (NaN, None)
null_counts = df_filtered.isnull().sum()

# 2. Count empty list representations (read as strings from CSV)
empty_list_counts = (df_filtered == '[]').sum()

# 3. Combine them to see the total missing/empty values per column
total_missing = null_counts + empty_list_counts

print("Total missing or '[]' values per column:")
display(total_missing)

Unique elements per column:


title                  2042
description            1990
level                    46
goal                    239
equipment                 3
program_length           18
time_per_workout         19
week                     18
day                      10
number_of_exercises      24
exercise_name          2666
sets                     18
reps                    190
intensity                11
created                2107
last_edit               516
dtype: int64

Total missing or '[]' values per column:


title                     0
description             710
level                  2578
goal                   2578
equipment                16
program_length           16
time_per_workout          0
week                      0
day                       0
number_of_exercises       0
exercise_name             0
sets                      0
reps                      0
intensity                 0
created                  16
last_edit                41
dtype: int64

In [41]:
# Get an array of all unique exercise names
unique_exercises = df_filtered['exercise_name'].dropna().unique()

# Print the total count
print(f"Total unique exercises: {len(unique_exercises)}\n")

# Display the full list
display(unique_exercises)

Total unique exercises: 2666



<StringArray>
[                  'Bench Press (Barbell)',
                 'Bent Over Row (Barbell)',
                'Shoulder Press (Machine)',
                            'Lat Pulldown',
                'Tricep Extension (Cable)',
                 'Preacher Curl (Barbell)',
                         'Squat (Barbell)',
                      'Stiff Leg Deadlift',
                               'Leg Press',
                                'Leg Curl',
 ...
                'Dual Handle Lat Pulldown',
             'Decline Machine Chest Press',
        'Super ROM Lateral Dumbbell Raise',
             'Medicine Ball Russian Twist',
             'Smith Machine Reverse Lunge',
            'Super ROM Overhead Cable Row',
                     'Dumbbell Calf Jumps',
         'Rear Delt 45 Degrees Cable Flye',
 'Triceps Diverging Pressdown (Long Rope)',
                  'Core Work (You Choose)']
Length: 2666, dtype: str

In [42]:
# Count the number of rows with negative reps values
num_negative_reps = (df_filtered['reps'] < 0).sum()

# Calculate the average reps value excluding the negative values
avg_reps_non_negative = df_filtered.loc[df_filtered['reps'] >= 0, 'reps'].mean()

print(f"Number of rows with negative reps: {num_negative_reps}")
print(f"Average reps (excluding negatives): {avg_reps_non_negative}")

Number of rows with negative reps: 22204
Average reps (excluding negatives): 10.792690785044767


In [43]:
# Print each equipment type and the number of rows that use it
print(df_filtered['equipment'].value_counts())

equipment
Full Gym         464904
At Home           29934
Dumbbell Only     13093
Name: count, dtype: int64


In [53]:
# Apply the filters
df_filtered = df_filtered[
    (df_filtered['equipment'] == "Full Gym")
].copy()
# Remove the specified columns
df_final = df_filtered.drop(columns=['description', 'created', 'last_edit'])

# Display the first few rows to verify they are gone
display(df_final.head())
# Display total rows
print(f"Total rows after filtering: {len(df_final)}")
#Solo equipment Full Gym, sin reps negativos, sin level vacíos o con '[]' como string.

,title,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity
325,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Bench Press (Barbell),5.0,9.0,8.0
326,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Bent Over Row (Barbell),5.0,9.0,8.0
327,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Shoulder Press (Machine),3.0,14.0,8.0
328,Lyle McDonald Routine (Strength/Hypertrophy Vers),"['Beginner', 'Novice', 'Intermediate', 'Advanc...","['Olympic Weightlifting', 'Muscle & Sculpting'...",Full Gym,12.0,90.0,1.0,1.0,6.0,Lat Pulldown,3.0,14.0,8.0
329,Lyle McDonald Routine (Strength/Hypertrophy Vers),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Tricep Extension (Cable),2.0,5.0,8.0


Total rows after filtering: 464904


In [54]:
# Display 10 random rows
df_final.sample(n=min(10, len(df_final)))

,title,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity
90601,Lex’s strength training,"['Novice', 'Intermediate']",['Bodybuilding'],Full Gym,12.0,60.0,4.0,3.0,9.0,Band Pull Apart,4.0,11.0,6.0
13084,REBUILD SYSTEM,"['Novice', 'Advanced']","['Bodybuilding', 'Muscle & Sculpting']",Full Gym,12.0,120.0,9.0,3.0,5.0,Romanian Deadlift (Barbell),2.0,11.0,7.0
82437,Startup Upper/Lower,['Novice'],"['Bodybuilding', 'Powerlifting', 'Muscle & Scu...",Full Gym,12.0,70.0,7.0,1.0,7.0,Seated Hamstring Curl,3.0,5.0,8.0
13693,MY PROGRAM (S3),['Advanced'],"['Powerbuilding', 'Bodybuilding', 'Athletics']",Full Gym,12.0,120.0,2.0,3.0,5.0,Chin-Up (Weighted),3.0,18.0,8.0
92808,Overall strength,"['Beginner', 'Novice', 'Intermediate']",['Bodyweight Fitness'],Full Gym,12.0,60.0,1.0,3.0,6.0,Back Extension,1.0,5.0,7.0
408243,Berserker,"['Intermediate', 'Advanced']","['Athletics', 'Powerlifting']",Full Gym,6.0,90.0,1.0,6.0,5.0,Bench Press (Barbell),3.0,10.0,7.0
311940,Beast Mode,"['Advanced', 'Intermediate']","['Bodybuilding', 'Powerbuilding']",Full Gym,10.0,50.0,6.0,6.0,5.0,Lateral Raise (Cable),3.0,10.0,8.0
80772,Hypertrophy V3,['Beginner'],['Muscle & Sculpting'],Full Gym,6.0,60.0,3.0,1.0,4.0,Lat Pulldown,4.0,12.0,9.0
94970,PPL+1: Precision Progression,"['Intermediate', 'Novice']","['Bodybuilding', 'Muscle & Sculpting']",Full Gym,13.0,90.0,11.0,1.0,5.0,Incline Bench Press (Dumbbell),3.0,3.0,8.0
595950,Tbjp inspired high intensity,['Advanced'],['Bodybuilding'],Full Gym,16.0,70.0,8.0,1.0,13.0,Chest Supported Row (Dumbbell),1.0,11.0,6.0


In [55]:
#check for null or negative values in sets, reps, intensity
import pandas as pd

# Create a summary table for the specific columns
summary_table = pd.DataFrame({
    'Null_Count': df_final[['sets', 'reps', 'intensity']].isnull().sum(),
    'Negative_Count': (df_final[['sets', 'reps', 'intensity']] < 0).sum()
})

# Display the table
display(summary_table)

,Null_Count,Negative_Count
sets,0,0
reps,0,20154
intensity,0,0


In [56]:
# Check what the negative values actually are to see if they are a code
print(df_final[df_final['reps'] < 0]['reps'].value_counts())

reps
-60.0      4153
-120.0     2165
-300.0     1427
-900.0     1349
-1800.0    1260
           ... 
-1620.0       1
-1740.0       1
-1980.0       1
-2040.0       1
-930.0        1
Name: count, Length: 122, dtype: int64


In [57]:
#Calculate Systemic Load per exercise
#S = sets * reps * intensity
import numpy as np

# Calculate systemic load: Sets * Reps * Intensity (but only if Reps >= 0. Otherwise, leave as NaN)
df_final['systemic_load'] = np.where(
    df_final['reps'] >= 0, 
    df_final['sets'] * df_final['reps'] * df_final['intensity'], 
    np.nan
)

# Display the result to verify
display(df_final[['sets', 'reps', 'systemic_load']].head())

,sets,reps,systemic_load
325,5.0,9.0,360.0
326,5.0,9.0,360.0
327,3.0,14.0,336.0
328,3.0,14.0,336.0
329,2.0,5.0,80.0


## 2. Data Aggregation (Exercise to Routine Level)

We aggregate the dataset using custom mapping rules (e.g. summing total weight/reps for systemic load, averaging intensities). This directly builds the input vectors required for the Keras `input_shape`.

**Academic Justification:** Deep Learning models require a uniform, structured input vector for each instance. Aggregating the sequential/tabular exercise steps into a single "routine" feature vector ensures the DNN receives the global physiological snapshot of the workout (volume, load, etc.) rather than disjointed exercises.

In [58]:
#Check if any routine has ALL exercises with missing or empty levels
import pandas as pd

def check_missing_levels_by_routine(df):
    """
    Groups the dataframe by routine ('title') and evaluates if ALL rows 
    within each routine have a missing or empty ('[]') level.
    
    Returns a dataframe with each routine and a True/False flag.
    """
    # 1. Create a boolean mask: True if level is NaN or the string representation is '[]'
    is_missing = df['level'].isna() | (df['level'].astype(str) == '[]')
    
    # 2. Attach this temporary check to the dataframe
    df_temp = df.assign(is_level_missing=is_missing)
    
    # 3. Group by title and use .all() 
    # .all() returns True ONLY if every single row in that group is True
    agg_df = df_temp.groupby('title')['is_level_missing'].all().reset_index()
    
    # Rename the column so it's easy to read
    agg_df = agg_df.rename(columns={'is_level_missing': 'all_levels_missing'})
    
    return agg_df

# --- 1. RUN THE CHECK ---
df_missing_check = check_missing_levels_by_routine(df_final)

# See the results
display(df_missing_check.head())

#See exactly WHICH routines have completely missing levels:
completely_missing_routines = df_missing_check[df_missing_check['all_levels_missing'] == True]

print(f"\nThere are {len(completely_missing_routines)} routines where EVERY exercise is missing a level.")
if len(completely_missing_routines) > 0:
    display(completely_missing_routines)

,title,all_levels_missing
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,False
1,(NOT MY PROGRAM)SHJ Jotaro,False
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,False
3,10 week deadlift focus,False
4,1000 lbs Club,False



There are 2 routines where EVERY exercise is missing a level.


,title,all_levels_missing
986,Mania (Upper/Lower),True
1727,Viking strong,True


In [59]:
def display_routine_rows(df, routine_title):
    """
    Filters the dataframe to show only the rows that belong to a specific routine.
    """
    # Filter the dataset where the 'title' exactly matches the requested routine
    routine_df = df[df['title'] == routine_title]
    
    print(f"Showing {len(routine_df)} rows for routine: '{routine_title}'")
    
    # Display the filtered dataframe
    display(routine_df)
    
    # (Optional) Return it in case you want to save it to a variable
    return routine_df


In [60]:
#Check the routine program 
my_routine_data = display_routine_rows(df_final, "Mania (Upper/Lower)")

Showing 288 rows for routine: 'Mania (Upper/Lower)'


,title,level,goal,equipment,program_length,time_per_workout,week,day,number_of_exercises,exercise_name,sets,reps,intensity,systemic_load
513536,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Squat (Barbell),3.0,15.0,7.0,315.0
513537,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Seated Row (Cable),2.0,12.0,7.0,168.0
513538,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Hamstring Curl,2.0,12.0,7.0,168.0
513539,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Standing Calf Raise,3.0,3.0,8.0,72.0
513540,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,1.0,1.0,6.0,Leg Extension,2.0,12.0,8.0,192.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
513819,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Chest Supported Row (Dumbbell),3.0,5.0,7.0,105.0
513820,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Tricep Pushdown (Cable),2.0,5.0,8.0,80.0
513821,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Incline Curl (Dumbbell),2.0,11.0,8.0,176.0
513822,Mania (Upper/Lower),[],[],Full Gym,12.0,90.0,12.0,4.0,7.0,Lateral Raise (Dumbbell),2.0,9.0,7.0,126.0


In [61]:
# --- 2. REMOVE THE BAD ROUTINES ---
# Extract the titles of the bad routines
bad_titles = completely_missing_routines['title']
# Filter df_final to keep only rows where the title is NOT (~) in the bad_titles list
df_final = df_final[~df_final['title'].isin(bad_titles)]
print(f"Successfully removed those {len(bad_titles)} routines from df_final")

Successfully removed those 2 routines from df_final


In [69]:
# --- GET WEEKLY VOLUME PER ROUTINE ---
def get_avg_weekly_volume(df):
    """
    Calculates the average weekly volume for each routine, 
    relying on the existing 'week' column.
    """
    # --- STEP 1: Get the volume per week ---
    # We sum() the volume to get the total volume of that routine for each week.
    weekly_totals = df.groupby(['title', 'week'])['sets'].sum().round(2).reset_index()
    
    # --- STEP 2: Get the average of those weeks ---
    # Now we group just by title, and get the mean() of those weekly totals
    avg_weekly_volume = weekly_totals.groupby('title')['sets'].mean().round(2).reset_index()
    
    # Rename the column so it's clear
    avg_weekly_volume = avg_weekly_volume.rename(columns={'sets': 'avg_weekly_volume'})
    
    return avg_weekly_volume


# --- How to use it ---
df_avg_weekly_volume = get_avg_weekly_volume(df_final)
#display 30 examples
display(df_avg_weekly_volume.head(10))

,title,avg_weekly_volume
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,54.00
1,(NOT MY PROGRAM)SHJ Jotaro,96.00
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,98.00
3,10 week deadlift focus,112.30
4,1000 lbs Club,108.00
5,10x3 Powerbuilding,62.67
6,12 WEEK FAT DESTROYER.,92.00
7,12 Week Amped Up,112.00
8,12 Week Women’s Bikini Body Program,54.00
9,12 Weeks Bikini Comp Ready,91.33


In [70]:
#-- GET DAILY VOLUME PER ROUTINE ---
def get_avg_daily_volume(df):
    """
    Calculates the average daily volume per routine, 
    where 'volume' is defined as the total number of sets.
    """
    # --- STEP 1: Get total sets per day ---
    # Group by title, week, and day, then sum() the sets to get the total sets done in that single session
    daily_totals = df.groupby(['title', 'week', 'day'])['sets'].sum().round(2).reset_index()
    
    # --- STEP 2: Get the average of those days ---
    # Now group just by title, and get the mean() of those daily session totals
    avg_daily_volume = daily_totals.groupby('title')['sets'].mean().round(2).reset_index()
    
    # Rename the column so it's clear we are measuring by sets
    avg_daily_volume = avg_daily_volume.rename(columns={'sets': 'avg_daily_volume_sets'})
    
    return avg_daily_volume

# --- How to use it ---
df_avg_daily_volume = get_avg_daily_volume(df_final)

display(df_avg_daily_volume.head(10))

,title,avg_daily_volume_sets
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,7.71
1,(NOT MY PROGRAM)SHJ Jotaro,24.00
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,19.60
3,10 week deadlift focus,23.40
4,1000 lbs Club,18.00
5,10x3 Powerbuilding,15.67
6,12 WEEK FAT DESTROYER.,23.00
7,12 Week Amped Up,18.67
8,12 Week Women’s Bikini Body Program,13.50
9,12 Weeks Bikini Comp Ready,13.05


In [71]:
# -- GET WEEKLY SYSTEMIC LOAD ---
def get_avg_weekly_systemic_load(df):
    """
    Calculates the average weekly systemic load for each routine.
    Assumes you have a column named 'systemic_load'.
    """
    # --- STEP 1: Get total systemic load per week ---
    # Group by title and week, then sum() to get the total load endured in that single week
    weekly_totals = df.groupby(['title', 'week'])['systemic_load'].sum().reset_index()
    
    # --- STEP 2: Get the average of those weeks ---
    # Group by title and get the mean() of those weekly totals
    avg_weekly_load = weekly_totals.groupby('title')['systemic_load'].mean().round(2).reset_index()
    
    # Rename the column to exactly what you requested
    avg_weekly_load = avg_weekly_load.rename(columns={'systemic_load': 'avg_systemic_load_per_week'})
    
    return avg_weekly_load

# --- How to use it ---
df_avg_systemic_load = get_avg_weekly_systemic_load(df_final)

display(df_avg_systemic_load.head(10))

,title,avg_systemic_load_per_week
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,4378.50
1,(NOT MY PROGRAM)SHJ Jotaro,5390.50
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,7805.33
3,10 week deadlift focus,9705.80
4,1000 lbs Club,8355.17
5,10x3 Powerbuilding,6173.08
6,12 WEEK FAT DESTROYER.,8555.58
7,12 Week Amped Up,6615.67
8,12 Week Women’s Bikini Body Program,2748.00
9,12 Weeks Bikini Comp Ready,7081.83


In [73]:
def get_routine_intensity_stats(df):
    """
    Calculates both the average intensity and the maximum intensity 
    for the entire routine.
    """
    # Group by routine title and calculate mean and max simultaneously
    intensity_stats = df.groupby('title').agg(
        avg_intensity=('intensity', 'mean'),
        max_intensity=('intensity', 'max')
    ).round(2).reset_index()
    
    return intensity_stats

# --- How to use it ---
df_intensity = get_routine_intensity_stats(df_final)

display(df_intensity.head(10))

,title,avg_intensity,max_intensity
0,(MASS MONSTER) High Intensity 4 Day Upper Lowe...,8.29,9.0
1,(NOT MY PROGRAM)SHJ Jotaro,7.10,8.0
2,1 PowerLift Per Day Powerbuilding 5 Day Bro Split,8.35,10.0
3,10 week deadlift focus,7.37,9.0
4,1000 lbs Club,7.95,9.0
5,10x3 Powerbuilding,7.53,9.0
6,12 WEEK FAT DESTROYER.,7.89,10.0
7,12 Week Amped Up,8.86,10.0
8,12 Week Women’s Bikini Body Program,8.18,9.0
9,12 Weeks Bikini Comp Ready,7.66,9.0


## 3. Save Processed Dataset

Save the aggregated DataFrame to a CSV so it can be uploaded to Google Colab for Phase 2.

In [ ]:
# df_processed.to_csv('../data/processed/aggregated_routines.csv', index=False)
print("Preprocessing completed. Proceed to Notebook 02 on Colab.")